**Interactive Juypter nootebook implementing a high-performance computation of $L(-k, \chi_D)$ via the Cohen–Eisenstein series.**     

We compute values of $L(-k, \chi_D)$ for fundamental discriminants in large intervals $[X, X + L]$ using half-integral weight Cohen–Eisenstein series Fourier coefficients in PARI/SageMath.

In [1]:
"""
Computation of negative special values L(-k, \chi_D) for fundamental discriminants
via Cohen-Eisenstein series coefficients in SageMath / PARI.

Author: Heiko Knospe, 2026
"""

def squarefree_mask(lo, hi):
    r"""
    Return a boolean list of length (hi - lo + 1) where index (n - lo) is True
    if and only if n is squarefree.
    """
    length = hi - lo + 1
    is_sq_free = [True] * length
    limit = int(QQ(hi).sqrt())

    # Standard prime sieve up to sqrt(hi)
    is_prime = [True] * (limit + 1)
    is_prime[0] = is_prime[1] = False
    for p in range(2, int(QQ(limit).sqrt()) + 1):
        if is_prime[p]:
            for m in range(p * p, limit + 1, p):
                is_prime[m] = False

    # Sieve square factors p^2 for prime p
    for p in range(2, limit + 1):
        if not is_prime[p]:
            continue
        p2 = p * p
        k = (lo + p2 - 1) // p2
        start = k * p2
        start_idx = start - lo
        if start_idx < length:
            for idx in range(start_idx, length, p2):
                is_sq_free[idx] = False

    return is_sq_free


def fundamental_discriminants_in_interval(X, L=10**6, sign=+1):
    r"""
    Compute fundamental discriminants D with |D| in [X, X + L] of the specified sign.

    Conditions for D to be fundamental:
      - If D > 0:
          * Odd:  D = n,  n == 1 (mod 4), n squarefree
          * Even: D = 4m, m == 2, 3 (mod 4), m squarefree
      - If D < 0:
          * Odd:  |D| = n,  n == 3 (mod 4), n squarefree
          * Even: |D| = 4m, m == 1, 2 (mod 4), m squarefree

    INPUT:
      - ``X``    -- Lower bound (|D| >= X >= 1).
      - ``L``    -- Interval length (default: 10^6).
      - ``sign`` -- Sign of discriminant (+1 or -1).
    """
    X = int(X)
    L = int(L)
    if X < 1:
        raise ValueError("X must be >= 1")
    if L < 0:
        raise ValueError("Interval length L must be >= 0")
    if sign not in (+1, -1):
        raise ValueError("sign must be +1 or -1")

    lo, hi = X, X + L
    Ds = []

    rem_odd = 1 if sign == +1 else 3
    rem_even = (2, 3) if sign == +1 else (1, 2)

    # 1. Odd fundamental discriminants |D| = n
    mask_n = squarefree_mask(lo, hi)
    start_n = lo + ((rem_odd - lo) % 4)
    for n in range(start_n, hi + 1, 4):
        if mask_n[n - lo]:
            Ds.append(sign * n)

    # 2. Even fundamental discriminants |D| = 4m
    mlo = (lo + 3) // 4
    mhi = hi // 4
    if mlo <= mhi:
        mask_m = squarefree_mask(mlo, mhi)
        for m in range(mlo, mhi + 1):
            if (m & 3) in rem_even and mask_m[m - mlo]:
                Ds.append(sign * 4 * m)

    Ds.sort(key=abs)
    return Ds


def L_block_negative_integers(k, X, L=10**6, sign=+1, include_zeros=False):
    r"""
    Compute list of tuples (D, L(-k, \chi_D)) for fundamental discriminants D
    of given sign with |D| in [X, X + L].

    Uses Cohen–Eisenstein series H_{m + 1/2} via PARI:
      m = k + 1, weight = m + 1/2.
      The coefficient of q^|D| equals L(1 - m, \chi_D) = L(-k, \chi_D)
      when (-1)^m * D > 0.
    """
    k = int(k)
    X = int(X)
    L = int(L)
    if k < 0:
        raise ValueError("k must be >= 0")
    if X < 1:
        raise ValueError("X must be >= 1")

    m = k + 1
    wt = QQ(2 * m + 1) / 2
    eps = +1 if (m % 2 == 0) else -1  # (-1)^m

    # Filter discriminants satisfying the non-vanishing parity condition
    Ds = fundamental_discriminants_in_interval(X, L=L, sign=sign)
    active_Ds = [D for D in Ds if eps * D > 0]
    if not active_Ds and not include_zeros:
        return []

    # Compute Fourier expansion up to hi = X + L
    hi = X + L
    H = pari.mfEH(wt)
    V = H.mfcoefs(hi)  # Retain PARI vector directly to minimize RAM overhead

    results = []
    for D in Ds:
        absD = abs(D)
        if eps * D <= 0:
            if include_zeros:
                results.append((ZZ(D), QQ(0)))
            continue
        # Convert single coefficient to Sage rational
        val = QQ(pari.lift(V[absD]))
        results.append((ZZ(D), val))

    return results

In [2]:
# Parameters
# allocate many gigabytes for high discriminants
# pari.allocatemem(16* 1024 * 1024 * 1024)

k = 12             # Value at s = -k (e.g., L(0, chi_D) related to class numbers)
X = 1              # Starting discriminant bound
L = 10**5          # Block size
sign = -1          # positive or negative fundamental discriminants 

# Execution
print(f"Computing L(-{k}, chi_D) for |D| in [{X}, {X + L}] with sign={sign}...")
vals = L_block_negative_integers(k=k, X=X, L=L, sign=sign)

print(f"Found {len(vals)} fundamental discriminants with non-zero L-values.")
print("First 10 values and comparison with direct computatation.")
for D, val in vals[:10]:
    print(f"  D = {D:10d} => L({-k}, chi_D) = {val}, check = {val==pari.lfun(D,-k)}")

# Save to disk
filename = f"L_vals_k{k}_sign{sign}_X{X}"
save(vals, filename)
print(f"Saved results to {filename}.sobj")
# vals=load(filename) to load the data

Computing L(-12, chi_D) for |D| in [1, 100001] with sign=-1...
Found 30392 fundamental discriminants with non-zero L-values.
First 10 values and comparison with direct computatation.
  D =         -3 => L(-12, chi_D) = 111202/3, check = True
  D =         -4 => L(-12, chi_D) = 2702765/2, check = True
  D =         -7 => L(-12, chi_D) = 1475050592, check = True
  D =         -8 => L(-12, chi_D) = 7828053417, check = True
  D =        -11 => L(-12, chi_D) = 419164502850, check = True
  D =        -15 => L(-12, chi_D) = 20240555475872, check = True
  D =        -19 => L(-12, chi_D) = 388503915123170, check = True
  D =        -20 => L(-12, chi_D) = 737740947722562, check = True
  D =        -23 => L(-12, chi_D) = 4233308426451936, check = True
  D =        -24 => L(-12, chi_D) = 7205584123783010, check = True
Saved results to L_vals_k12_sign-1_X1.sobj
